In [ ]:
# RTB_BOOTSTRAP_CELL -- sets up the environment (Colab / JupyterLite / local install); click to expand. Generated from docs/notebooks/_rtb_nb_bootstrap.py by sync_bootstrap.py -- do not hand-edit.

"""Environment bootstrap for roboticstoolbox-python's Jupyter notebooks.

Installs the toolbox (and reports how) across the three environments a notebook in
this folder might run in: a local Jupyter/VS Code install, Google Colab, and
JupyterLite (Pyodide/WASM, in-browser).

This file is the single source of truth for that logic. Every notebook's own
bootstrap cell is a generated copy of this file's content, produced by
sync_bootstrap.py -- see docs/notebooks/README.md for the full explanation.
"""

import subprocess
import sys
from pathlib import Path


async def ensure_installed() -> bool:
    """Install roboticstoolbox-python if needed, and report the environment.

    :returns: True if running on Google Colab, False otherwise.
    """
    if sys.platform == "emscripten":
        import micropip

        # spatialmath-python <= 1.1.18 lists pre-commit (a dev tool) as a runtime
        # dependency; its virtualenv requires packaging>=26.3, which conflicts with
        # the packaging 24.2 built into Pyodide, so a normal resolve fails.  Install
        # it without dependencies first: it is then already satisfied when the
        # toolbox's own dependencies are resolved below.  Remove this line once a
        # spatialmath release without that dependency is out
        # (rai-opensource/spatialmath-python#250).
        await micropip.install("spatialmath-python", deps=False)
        await micropip.install("ipympl")

        wheels = sorted(Path("/pypi").glob("roboticstoolbox_python-*.whl"))
        if wheels:
            # Prefer the wheel bundled with this JupyterLite site. Relative,
            # not absolute: a leading slash resolves against the origin, not
            # this site's own base URL (a GitHub Pages project subpath).
            await micropip.install(f"pypi/{wheels[-1].name}")
        else:
            # Fall back to PyPI when running outside the published site layout.
            await micropip.install("roboticstoolbox-python")
        where, colab = "in browser", False
    else:
        try:
            import google.colab  # noqa: F401
        except ImportError:
            where, colab = "locally", False
        else:
            print("Installing roboticstoolbox-python...")
            subprocess.run(
                [
                    sys.executable,
                    "-m",
                    "pip",
                    "install",
                    "-q",
                    "roboticstoolbox-python",
                ],
                check=True,
            )
            where, colab = "on Colab", True

    import roboticstoolbox

    version = getattr(roboticstoolbox, "__version__", "unknown")
    print(f"Running {where} using RTB v{version}")
    return colab

COLAB = await ensure_installed()


In [ ]:
# We will do the imports required for this notebook here

# numpy provides import array and linear algebra utilities
import numpy as np

# the robotics toolbox provides robotics specific functionality
import roboticstoolbox as rtb

# ansitable is a great package for printing tables in a terminal
from ansitable import ANSITable

# a package for creating dynamic progress bars
from progress.bar import Bar

# swift is a lightweight browser-based simulator which comes with the toolbox
from swift import Swift

# spatialgeometry is a utility package for dealing with geometric objects
import spatialgeometry as sg

# provides sleep functionaly
import time

# suppress warnings
import warnings

warnings.filterwarnings("ignore")

# Numerical Inverse Kinematics Benchmark

<br>

In this Notebook we benchmark the performance of each numerical IK solver we provide with the toolbox.

There are several ways to use IK solvers with the Toolbox:

* The **fast** methods which are implemented in C++ and wrapped in Python. These methods are available from ``robot`` and ``ets`` classes and start with ``ik_``. For example ``ik_LM`` is a fast C++ based solver.

* The **slow** methods which are implemented in Python. These methods are available from ``robot`` and ``ets`` classes and start with ``ikine_``. For example ``ikine_LM`` is a slow Python based solver.

* The **slow** class-based operation which is implemented in Python. These classes inherit from the ``IKSolver`` class and start with ``IK_``. For example ``IK_LM`` is a class which provides a slow Python based solver.

### Contents

[1.0 Fast IK Methods](#1)

[2.0 Python IK Methods](#2)

[3.0 Python Class Based IK Methods](#3)



<br>

<a id='1'></a>

### 1.0 Fast IK Methods
---

In [ ]:
# Our robot and ETS
robot = rtb.models.Panda()
ets = robot.ets()

In [ ]:
# Number of problems to solve
speed_problems = 10000

# Cartesion DoF priority matrix
mask = np.array([1.0, 1.0, 1.0, 1.0, 1.0, 1.0])

# random valid q values which will define Tep
q_rand = ets.random_q(speed_problems)

# Our desired end-effector poses
Tep = np.zeros((speed_problems, 4, 4))

for i in range(speed_problems):
    Tep[i] = ets.eval(q_rand[i])

# Maximum iterations allowed in a search
ilimit = 30

# Maximum searches allowed per problem
slimit = 100

# Solution tolerance
tol = 1e-6

joint_limits = False

In [ ]:
speed_solvers = [
    lambda Tep: ets.ik_NR(
        Tep,
        ilimit=ilimit,
        slimit=slimit,
        tol=tol,
        mask=mask,
        pinv=True,
        joint_limits=joint_limits,
    ),
    lambda Tep: ets.ik_GN(
        Tep,
        ilimit=ilimit,
        slimit=slimit,
        tol=tol,
        mask=mask,
        pinv=True,
        joint_limits=joint_limits,
    ),
    lambda Tep: ets.ik_LM(
        Tep,
        ilimit=ilimit,
        slimit=slimit,
        tol=tol,
        mask=mask,
        k=1e-4,
        joint_limits=joint_limits,
        method="wampler",
    ),
    lambda Tep: ets.ik_LM(
        Tep,
        q0=None,
        ilimit=ilimit,
        slimit=slimit,
        tol=tol,
        mask=mask,
        k=0.1,
        joint_limits=joint_limits,
        method="chan",
    ),
    lambda Tep: ets.ik_LM(
        Tep,
        ilimit=ilimit,
        slimit=slimit,
        tol=tol,
        mask=mask,
        k=0.0001,
        joint_limits=joint_limits,
        method="sugihara",
    ),
]

speed_names = [
    "Newton Raphson (pinv=True)",
    "Gauss Newton (pinv=True)",
    "LM Wampler 1e-4",
    "LM Chan 0.1",
    "LM Sugihara 0.0001",
]

times = []

In [ ]:
for name, solver in zip(speed_names, speed_solvers):
    print(f"Solving with {name}")

    start = time.time()

    for i in range(speed_problems):
        solver(Tep[i])

    total_time = time.time() - start
    times.append(total_time)


print(
    f"\nNumerical Inverse Kinematics Methods Times Compared over {speed_problems} problems\n"
)

table = ANSITable(
    "Method",
    "Total Time (s)",
    "Average Time per Solution (μs)",
    border="thin",
)

for name, t in zip(speed_names, times):
    table.row(
        name,
        np.round(t, 4),
        np.round((t / speed_problems) * 1e6, 4),
    )

table.print()